# ADBE Economic Compounding

Slice 2A asked, one year at a time, "did the economics improve this year?" This notebook asks the longer question: **over a multi-year period, how fast did Adobe compound its economics per share, and how good was the quality of that compounding?**

Everything here is deterministic and built only on the trusted Feature 1 fundamentals and the Feature 2A annual snapshots. No new SEC data is fetched, and the classification is a documented rule with named drivers, never an opaque score.

The one idea to hold onto: **CAGR is computed over fiscal-year *intervals*, not observations, and a period is named by its interval count.** FY2021 through FY2025 has five annual observations but only **four** compounding intervals, so it is a **4-year CAGR**. A true **3-year CAGR** is FY2022 → FY2025. Naming a 3-observation window (FY2023 → FY2025) a "3-year CAGR" would be wrong — that is a 2-year CAGR.

We answer eight questions:

1. How fast did revenue compound?
2. How fast did total FCF compound?
3. How fast did FCF per share compound?
4. How much did a shrinking share count contribute to the difference?
5. Did margins improve or deteriorate?
6. Did ROIC improve or deteriorate?
7. Was the recent 3-year period stronger or weaker than the longer period?
8. What does OwnerLens conclude about Adobe as a compounding machine?

> Set an SEC User-Agent identifying your application and an administrative contact. Replace the sample contact below before making live requests.

In [7]:
import os

from owner_lens import SecClient, compounding_views_from_facts, format_compounding_view

USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

raw = SecClient(USER_AGENT).retrieve_company_facts("ADBE").raw_facts
recent, long_term = compounding_views_from_facts(raw)
(recent.start_fiscal_year, recent.end_fiscal_year, recent.years), (
    long_term.start_fiscal_year,
    long_term.end_fiscal_year,
    long_term.years,
)

((2023, 2025, 2), (2021, 2025, 4))

## The longer-run picture: the longest available CAGR

Start here for the durable economics. The header names the period by its interval count (its N-year CAGR), which anchors every rate.

In [9]:
print(format_compounding_view(long_term))

ADBE Economic Compounding — FY2021 → FY2025  (4 intervals)

  Revenue CAGR:       +10.8%
  FCF CAGR:           +9.4%
  FCF/share CAGR:     +12.7%
  Diluted-share CAGR: -2.9%

  Operating margin:   36.8% → 36.6%
  FCF margin:         43.6% → 41.4%
  ROIC:               40.0% → 61.6%
  Net cash/debt:      +1,675,000,000 → +385,000,000

  Annual snapshots:   IMPROVING 4  STABLE 0  DETERIORATING 0  INSUFFICIENT 1

  Classification:     COMPOUNDING
  Drivers:
    - MODERATE_FCF_PER_SHARE_COMPOUNDING
    - AGGREGATE_FCF_GREW
    - ROIC_HIGH_AND_SUSTAINED
    - ROIC_IMPROVED
    - FCF_MARGIN_CONTRACTED
    - SHARE_COUNT_SHRANK
    - BALANCE_SHEET_DETERIORATED
    - CONSISTENT_ANNUAL_IMPROVEMENT


## Questions 1-4: aggregate growth versus per-share compounding

Revenue and total FCF describe the *business*. FCF per share describes the *owner*. The gap between total-FCF CAGR and FCF-per-share CAGR is what the share count did: a negative diluted-share CAGR (buybacks) lifts per-share compounding above aggregate compounding.

In [10]:
def pct(value):
    return f"{value * 100:+.1f}%" if value is not None else "n/a"


for view in (long_term, recent):
    lift = None
    if view.fcf_per_share_cagr is not None and view.fcf_cagr is not None:
        lift = view.fcf_per_share_cagr - view.fcf_cagr
    print(f"{view.years}-year CAGR  FY{view.start_fiscal_year}-{view.end_fiscal_year}")
    print(f"    revenue CAGR:       {pct(view.revenue_cagr)}")
    print(f"    total FCF CAGR:     {pct(view.fcf_cagr)}")
    print(f"    FCF/share CAGR:     {pct(view.fcf_per_share_cagr)}")
    print(f"    diluted-share CAGR: {pct(view.diluted_share_cagr)}")
    print(f"    per-share lift from share count: {pct(lift)}")

4-year CAGR  FY2021-2025
    revenue CAGR:       +10.8%
    total FCF CAGR:     +9.4%
    FCF/share CAGR:     +12.7%
    diluted-share CAGR: -2.9%
    per-share lift from share count: +3.3%
2-year CAGR  FY2023-2025
    revenue CAGR:       +10.7%
    total FCF CAGR:     +19.1%
    FCF/share CAGR:     +23.5%
    diluted-share CAGR: -3.6%
    per-share lift from share count: +4.4%


## Questions 5-6: margins and capital efficiency

Growth is only half the story. A period compounds *well* when margins hold or expand and capital efficiency stays high. ROIC is shown start → end so you can see the trajectory, not just the endpoint.

In [11]:
def lvl(value):
    return f"{value * 100:.1f}%" if value is not None else "n/a"


for view in (long_term, recent):
    print(f"{view.years}-year CAGR")
    print(f"    operating margin: {lvl(view.operating_margin_start)} → {lvl(view.operating_margin_end)}  ({pct(view.operating_margin_change)})")
    print(f"    FCF margin:       {lvl(view.fcf_margin_start)} → {lvl(view.fcf_margin_end)}  ({pct(view.fcf_margin_change)})")
    print(f"    ROIC:             {lvl(view.roic_start)} → {lvl(view.roic_end)}  ({pct(view.roic_change)})")

4-year CAGR
    operating margin: 36.8% → 36.6%  (-0.1%)
    FCF margin:       43.6% → 41.4%  (-2.1%)
    ROIC:             40.0% → 61.6%  (+21.6%)
2-year CAGR
    operating margin: 34.3% → 36.6%  (+2.4%)
    FCF margin:       35.8% → 41.4%  (+5.7%)
    ROIC:             43.5% → 61.6%  (+18.0%)


## Question 7: recent momentum versus the longer run

This is where 2B earns its keep. If the recent 3-year classification is stronger than the longer-run one, FY2025 is not a one-off — recent economics are accelerating. If it is weaker, a strong long-run record is fading. The annual snapshot counts show whether the verdict rests on consistent improvement or on mixed years.

In [12]:
print(format_compounding_view(recent))

ADBE Economic Compounding — FY2023 → FY2025  (2 intervals)

  Revenue CAGR:       +10.7%
  FCF CAGR:           +19.1%
  FCF/share CAGR:     +23.5%
  Diluted-share CAGR: -3.6%

  Operating margin:   34.3% → 36.6%
  FCF margin:         35.8% → 41.4%
  ROIC:               43.5% → 61.6%
  Net cash/debt:      +4,208,000,000 → +385,000,000

  Annual snapshots:   IMPROVING 3  STABLE 0  DETERIORATING 0  INSUFFICIENT 0

  Classification:     STRONGLY_COMPOUNDING
  Drivers:
    - STRONG_FCF_PER_SHARE_COMPOUNDING
    - AGGREGATE_FCF_GREW
    - ROIC_HIGH_AND_SUSTAINED
    - ROIC_IMPROVED
    - OPERATING_MARGIN_EXPANDED
    - FCF_MARGIN_EXPANDED
    - SHARE_COUNT_SHRANK
    - BALANCE_SHEET_DETERIORATED
    - CONSISTENT_ANNUAL_IMPROVEMENT


In [13]:
for view in (recent, long_term):
    codes = ", ".join(driver.value for driver in view.drivers) or "none"
    print(f"{view.years}-year CAGR: {view.classification.value}")
    print(f"    improving={view.improving_count} stable={view.stable_count} deteriorating={view.deteriorating_count} insufficient={view.insufficient_count}")
    print(f"    {codes}")

2-year CAGR: STRONGLY_COMPOUNDING
    improving=3 stable=0 deteriorating=0 insufficient=0
    STRONG_FCF_PER_SHARE_COMPOUNDING, AGGREGATE_FCF_GREW, ROIC_HIGH_AND_SUSTAINED, ROIC_IMPROVED, OPERATING_MARGIN_EXPANDED, FCF_MARGIN_EXPANDED, SHARE_COUNT_SHRANK, BALANCE_SHEET_DETERIORATED, CONSISTENT_ANNUAL_IMPROVEMENT
4-year CAGR: COMPOUNDING
    improving=4 stable=0 deteriorating=0 insufficient=1
    MODERATE_FCF_PER_SHARE_COMPOUNDING, AGGREGATE_FCF_GREW, ROIC_HIGH_AND_SUSTAINED, ROIC_IMPROVED, FCF_MARGIN_CONTRACTED, SHARE_COUNT_SHRANK, BALANCE_SHEET_DETERIORATED, CONSISTENT_ANNUAL_IMPROVEMENT


## Question 8: what does OwnerLens conclude?

Read the two classifications together. A recent 3-year verdict that is at least as strong as the longer-run verdict, backed by consistent annual improvement, sustained high ROIC, and a shrinking share count, is the signature of a durable per-share compounding machine — not a single lucky year.

What this view is **not**: it is not intrinsic value, a valuation, an expected return, or a score. FCF-per-share CAGR is treated as the primary *observable* contributor to owner economic-value growth, and the thresholds that separate "strong" from "noise" are centralized, named, and intentionally imprecise — meant to be economically intuitive and easy to revisit, not tuned to make Adobe look good.